# 16 — Interpreting Transformer Representations

**Description:** Train a tiny sparse autoencoder on synthetic superposed activations, inspect its recovered feature dictionary, and connect feature discovery to attention and MLP circuits.
**Level:** Beginner
**Tags:** Language Models, Mechanistic Interpretability, Sparse Autoencoders, Features, PyTorch

Notebook 15 showed why individual neuron coordinates may mix many features. Mechanistic interpretability asks how model computations produce behavior. One tool is a **sparse autoencoder (SAE)**, which learns a larger dictionary of feature directions and represents each activation using only a few of them.

This notebook trains a small SAE on synthetic activations whose true features we know. That lets us evaluate recovery honestly before discussing the harder case of real Transformer activations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

torch.manual_seed(16)
np.random.seed(16)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Create known features in superposition

We hide 12 unit feature directions inside an 8D activation space. Each example activates two features, so the data is sparse even though each observed activation is dense.

In [ ]:
rng = np.random.default_rng(16)
d_activation = 8
true_features = 12
examples = 4000
active_per_example = 2

true_dictionary = rng.normal(size=(true_features, d_activation))
true_dictionary /= np.linalg.norm(true_dictionary, axis=1, keepdims=True)

true_codes = np.zeros((examples, true_features), dtype=np.float32)
for row in range(examples):
    active = rng.choice(true_features, size=active_per_example, replace=False)
    true_codes[row, active] = rng.uniform(0.5, 1.5, size=active_per_example)

activations = true_codes @ true_dictionary
activations += 0.01 * rng.normal(size=activations.shape)
activations = activations.astype(np.float32)

print("true codes: ", true_codes.shape)
print("dictionary: ", true_dictionary.shape)
print("activations:", activations.shape)
print("fraction of active true features:", np.mean(true_codes > 0))

The learner sees only the dense 8D activations, not the true codes or dictionary.

## 2. An autoencoder reconstructs its input

An encoder maps an activation $x$ to a wider latent code $z$. A decoder reconstructs $x$ from the code:

$$z=\operatorname{ReLU}(W_{enc}x+b), \qquad \hat{x}=W_{dec}z$$

Without a sparsity constraint, the latent units need not align with the sparse generating features.

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, d_input, d_features):
        super().__init__()
        self.encoder = nn.Linear(d_input, d_features)
        self.decoder = nn.Linear(d_features, d_input, bias=False)

    def forward(self, x):
        code = torch.relu(self.encoder(x))
        reconstruction = self.decoder(code)
        return reconstruction, code

model = SparseAutoencoder(d_activation, true_features)
print(model)

## 3. Reconstruction plus sparsity

The training objective balances two goals:

$$\mathcal{L}=\underbrace{\|x-\hat{x}\|_2^2}_{reconstruction}+\lambda\underbrace{\|z\|_1}_{sparsity}$$

The L1 term encourages most feature activations to remain near zero. The coefficient $\lambda$ controls the tradeoff.

In [ ]:
data = torch.tensor(activations)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
sparsity_coefficient = 0.1
history = {"loss": [], "reconstruction": [], "sparsity": []}

for step in range(800):
    reconstruction, learned_codes = model(data)
    reconstruction_loss = torch.mean((reconstruction - data) ** 2)
    sparsity_loss = torch.mean(torch.abs(learned_codes))
    loss = reconstruction_loss + sparsity_coefficient * sparsity_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Remove the arbitrary scale tradeoff between encoder and decoder.
    with torch.no_grad():
        norms = model.decoder.weight.norm(dim=0, keepdim=True).clamp_min(1e-8)
        model.decoder.weight.div_(norms)
        model.encoder.weight.mul_(norms.T)
        model.encoder.bias.mul_(norms.squeeze(0))

    if step % 10 == 0:
        history["loss"].append(loss.item())
        history["reconstruction"].append(reconstruction_loss.item())
        history["sparsity"].append(sparsity_loss.item())

print("final reconstruction MSE:", history["reconstruction"][-1])
print("final mean |code|:       ", history["sparsity"][-1])

In [ ]:
steps = np.arange(len(history["loss"])) * 10
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(steps, history["reconstruction"])
axes[0].set(xlabel="training step", ylabel="MSE", title="Reconstruction improves")
axes[1].plot(steps, history["sparsity"], color="#F58518")
axes[1].set(xlabel="training step", ylabel="mean absolute code", title="Latent activity remains penalized")
plt.show()

## 4. Measure sparsity and reconstruction

In [ ]:
with torch.no_grad():
    reconstructed, learned_codes = model(data)

relative_error = torch.linalg.norm(reconstructed - data) / torch.linalg.norm(data)
active_fraction = (learned_codes > 0.05).float().mean()
active_per_row = (learned_codes > 0.05).float().sum(dim=1).mean()

print("relative reconstruction error:", relative_error.item())
print("fraction of active SAE units:  ", active_fraction.item())
print("active SAE units per example:  ", active_per_row.item())

A useful SAE needs both reasonable reconstruction and sparse codes. Optimizing only one side gives a trivial failure: dense perfect copying or all-zero features.

## 5. Did the SAE recover the true directions?

SAE feature order is arbitrary. Compare every normalized decoder direction with every true direction using absolute cosine similarity, then find each learned feature's best match.

In [ ]:
learned_dictionary = model.decoder.weight.detach().T.numpy()
learned_dictionary /= np.linalg.norm(learned_dictionary, axis=1, keepdims=True)
similarity = np.abs(learned_dictionary @ true_dictionary.T)
best_matches = similarity.argmax(axis=1)
best_scores = similarity.max(axis=1)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(similarity, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set(xlabel="true synthetic feature", ylabel="learned SAE feature", title="Absolute cosine similarity between dictionaries")
ax.set_xticks(range(true_features))
ax.set_yticks(range(true_features))
fig.colorbar(image, ax=ax, label="|cosine similarity|")
plt.show()

for learned, (true, score) in enumerate(zip(best_matches, best_scores)):
    print(f"learned {learned:2d} -> true {true:2d}, |cos|={score:.3f}")

Good diagonal structure may appear only after permuting rows because feature labels are exchangeable. Not every run recovers every feature perfectly: optimization, noise, correlated features, dead units, and the sparsity coefficient all matter.

## 6. Inspect examples that activate one learned feature

Interpretability work often starts by finding dataset examples with the highest activation for a feature.

In [ ]:
feature_id = int(np.argmax(best_scores))
top_examples = torch.topk(learned_codes[:, feature_id], k=5).indices.numpy()
matched_true = best_matches[feature_id]

print(f"learned feature {feature_id} best matches true feature {matched_true}")
for index in top_examples:
    actual = np.flatnonzero(true_codes[index] > 0).tolist()
    strength = learned_codes[index, feature_id].item()
    print(f"example {index:4d}: SAE activation={strength:.3f}, true active features={actual}")

Because this dataset is synthetic, we can compare top examples with ground truth. In a language model, researchers instead inspect activating text, test interventions, and trace downstream effects.

## 7. Change the sparsity strength

The central SAE tradeoff is empirical:

- too little sparsity can produce dense, hard-to-label codes;
- too much sparsity can miss information and harm reconstruction;
- feature count, data coverage, architecture, and training also matter.

Try `0.0`, `0.01`, or `0.1` and compare reconstruction and active units.

## 8. From features to circuits

Discovering a feature is only a beginning. A mechanistic explanation asks how components interact:

```text
earlier residual features
        ↓
attention heads route information between positions
        ↓
MLP detectors transform and write features
        ↓
later residual features
        ↓
logits and model behavior
```

SAEs can provide a feature basis for analyzing these paths. Causal interventions—adding, removing, or patching activations—help test whether a proposed feature actually affects behavior.

## 9. Responsible interpretation

- A high-activating example suggests a hypothesis; it does not establish what a feature “means.”
- SAE features depend on the dataset, layer, hyperparameters, and random initialization.
- Reconstruction is imperfect, so some model information may be missing.
- Features can split, merge, or remain uninterpretable.
- Correlation is not causation; interventions strengthen mechanistic claims.

Sparse autoencoders are a tool for analysis, not an automatic solution to interpretability.

## 10. Challenges

1. Train with no L1 penalty and compare latent density.
2. Increase the dictionary to 24 learned features. Do true features split across units?
3. Make two true directions highly correlated and inspect recovery.
4. Hold out 20% of activations and report test reconstruction error.
5. Zero one learned feature during reconstruction and measure which examples change most.
6. Replace synthetic activations with hidden states captured from a tiny trained model.

## How Transformers Work: complete

Across Notebooks 01–16, we followed the whole inference path:

```text
text → tokens → embeddings → attention → contextual values
     → multi-head output → MLP feature transformation → residual stream
     → logits → probabilities → generated tokens
```

We also developed a careful interpretation lens: knowledge can be distributed, features may live in superposition, individual neurons can be polysemantic, and sparse dictionaries can help reveal candidate features.

## Takeaways

- Sparse autoencoders reconstruct dense activations through a wider, sparse feature code.
- Reconstruction loss preserves information; an L1 penalty encourages sparse feature use.
- Synthetic data with known features lets us measure dictionary recovery directly.
- Feature discovery is not a complete explanation: circuits connect features through attention, MLPs, and residual streams.
- Interpretations need validation with diverse examples and causal interventions.
- The next chapter can now move from how Transformers compute to how their parameters learn: loss, gradients, optimization, and pretraining.